# Experiment 11: PolyGuardPrompts + M-ALERT Multilingual Safety (E4)

**Reviewer concern (R1, R2):** the original §14.4 multilingual experiment used 15 self-translated prompts per language, which R1 said is too small to support cross-language ordering claims.

**This notebook:** evaluates the same 4 SLMs (Qwen 2.5-3B, Llama 3.2-3B, Qwen 3-4B, Phi-4-Mini) on:

- **PolyGuardPrompts** (`ToxicityPrompts/PolyGuardPrompts`, single `test` split, filtered by   the flat `language` column and `prompt_harm_label == 'harmful'`) at 100 prompts/language for   English, Arabic, Chinese, Spanish. PolyGuardPrompts does **not** include Romanian; we   preserve the small self-translated Romanian probe in §14.4 as the survey's only Romanian   evidence, and the absence of Romanian from PolyGuardPrompts itself confirms the §9 claim   that Eastern European languages remain uncovered by major multilingual safety benchmarks.
- **M-ALERT** (`felfri/M-ALERT`, single `train` split where each row carries the same prompt   in all 5 languages) at 100 paired prompts spanning EN, DE, FR, IT, ES.

Reports ASR (= harmful response rate per Llama Guard 3-1B) with 95% Wilson CIs per (model, dataset, lang).

**Output:** `experiments/results/multilingual_polyguard_malert.json` and a CSV summary.

**Runtime:** ~2-3 h on A100 (4 models × ~1000 generations + ~1000 judgments).

**Schema notes** (verified against the dataset cards on 2026-05-28):
- PolyGuardPrompts has flat columns: `prompt`, `response`, `prompt_harm_label`,   `response_refusal_label`, `response_harm_label`, `subcategory`, `language`, `adversarial`,   `id`, plus a few `*_agreement` floats and `prompt_categories`/`response_categories` strings.   We use `prompt` and filter on `prompt_harm_label == 'harmful'` and `language == lang`.   (The dataset card's 'Data Fields' section claimed a nested `metadata` dict; the actual   parquet schema returned by `huggingface.co/api/datasets` is flat. Verified 2026-05-28.)
- M-ALERT has fields: `id`, `en`, `de`, `es`, `fr`, `it`, `category`. Each row is one prompt   translated into all 5 languages — same row read across columns gives parallel multilingual   prompts, which is methodologically stronger than independent per-language samples.
- PolyGuardPrompts and M-ALERT are both **public** (no gating) per the HF dataset API.   Only Llama Guard 3-1B (the judge) is gated; the HF login cell below covers it.


## Setup

In [ ]:
%%capture
!pip install -U 'transformers>=4.51' 'accelerate>=1.1' huggingface_hub datasets ipywidgets statsmodels -q
import os, json, time, gc, hashlib, random
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Hard-fail if Colab didn't allocate a GPU ---
assert torch.cuda.is_available(), (
    'No GPU detected. In Colab: Runtime > Change runtime type > A100 GPU. '
    'This notebook will not run on CPU.'
)
_gpu_name = torch.cuda.get_device_name(0)
_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'transformers: {transformers.__version__}')
print(f'GPU: {_gpu_name}')
print(f'VRAM: {_vram_gb:.1f} GB')
if 'A100' not in _gpu_name and _vram_gb < 35:
    print()
    print('WARNING: this notebook is configured for an A100 (40 GB).')
    print(f'You appear to have a {_gpu_name} with {_vram_gb:.1f} GB.')
    print('Smaller GPUs may OOM on the 13B HarmBench classifier or the GCG run.')
    print('Consider switching to A100 via Runtime > Change runtime type.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PhD/paper1-survey')
RESULTS_DIR = DRIVE_ROOT / 'experiments' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)


In [ ]:
# Reproducibility config (do not edit per-run; logged with results)
REPRO = {
    'notebook': None,  # set per-notebook
    'created': datetime.utcnow().isoformat() + 'Z',
    'transformers': transformers.__version__,
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': 42,
}
random.seed(REPRO['seed'])
np.random.seed(REPRO['seed'])
torch.manual_seed(REPRO['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(REPRO['seed'])


In [ ]:
REPRO['notebook'] = '11_polyguard_multilingual'
REPRO['polyguard_dataset'] = 'ToxicityPrompts/PolyGuardPrompts'
REPRO['malert_dataset'] = 'felfri/M-ALERT'
# PolyGuardPrompts uses full language names ('arabic', 'chinese', ...). It covers 17
# languages but Romanian is NOT one of them - this is itself a finding the §9 narrative
# already cites. Romanian stays in the small self-translated probe (§14.4) as the only
# Romanian evidence in the survey.
REPRO['polyguard_languages'] = ['english', 'arabic', 'chinese', 'spanish']
REPRO['malert_languages'] = ['en', 'de', 'fr', 'it', 'es']
REPRO['n_per_language'] = 100
REPRO['judge'] = 'meta-llama/Llama-Guard-3-1B'
REPRO['models'] = [
    'Qwen/Qwen2.5-3B-Instruct',
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen3-4B',
    'microsoft/Phi-4-mini-instruct',
]
REPRO['max_new_tokens'] = 256
REPRO


## Authenticate with HuggingFace

Llama Guard 3-1B (the judge) is gated. Two options:

1. **Colab Secrets (recommended).** In the left sidebar, click the key icon, add `HF_TOKEN`    with the value of your token from https://huggingface.co/settings/tokens, and toggle    notebook access on. The login cell below will pick it up silently.
2. **Manual paste.** If no Colab Secret is set, the cell falls back to a `getpass` prompt.

Either way the token only needs `read` scope. PolyGuardPrompts and M-ALERT are public.


In [ ]:
from huggingface_hub import login

_token = os.environ.get('HF_TOKEN')
if not _token:
    try:
        from google.colab import userdata
        _token = userdata.get('HF_TOKEN')
        if _token:
            print('Using HF_TOKEN from Colab Secrets.')
    except Exception:
        _token = None
if not _token:
    from getpass import getpass
    _token = getpass('HuggingFace token (or set HF_TOKEN in Colab Secrets): ').strip()
os.environ['HF_TOKEN'] = _token
login(token=_token, add_to_git_credential=False)
print('Logged in.')


## Sample harmful prompts from both datasets

PolyGuardPrompts: filter `prompt_harm_label == 'harmful'`, group by the flat `language` column, take 100 per language with a fixed seed.

M-ALERT: sample 100 ids from the train split (the same ids across all 5 languages give parallel prompts), then read each language column.


In [ ]:
from datasets import load_dataset
import pandas as pd

# --- PolyGuardPrompts ---
pg = load_dataset(REPRO['polyguard_dataset'], split='test')
pg_df = pg.to_pandas()
print('PolyGuardPrompts columns:', list(pg_df.columns))
# Defensive: dataset card showed a nested 'metadata' dict, but the actual parquet has
# a flat 'language' column. Pick whichever is present.
if 'language' in pg_df.columns:
    pg_df['lang'] = pg_df['language'].astype(str).str.lower()
elif 'metadata' in pg_df.columns:
    pg_df['lang'] = pg_df['metadata'].apply(lambda m: (m or {}).get('language', '').lower())
else:
    raise RuntimeError('Could not find a language column on PolyGuardPrompts.')
pg_df = pg_df[pg_df['prompt_harm_label'].astype(str).str.lower() == 'harmful']
print('PolyGuardPrompts harmful count by language (top 20):')
print(pg_df.groupby('lang').size().sort_values(ascending=False).head(20))

pg_samples = []
for lang in REPRO['polyguard_languages']:
    sub = pg_df[pg_df['lang'] == lang]
    if len(sub) == 0:
        print(f'WARNING: PolyGuardPrompts has 0 harmful prompts for lang={lang}')
        continue
    sub = sub.sample(
        n=min(REPRO['n_per_language'], len(sub)),
        random_state=REPRO['seed'],
    )
    sub = sub[['prompt']].copy()
    sub['lang'] = lang
    sub['dataset'] = 'polyguard'
    pg_samples.append(sub.reset_index(drop=True))
pg_samples = pd.concat(pg_samples, ignore_index=True)
print(f'PolyGuardPrompts: sampled {len(pg_samples)} prompts total')


In [ ]:
# --- M-ALERT ---
ma = load_dataset(REPRO['malert_dataset'], split='train')
ma_df = ma.to_pandas()
print(f'M-ALERT total parallel prompts: {len(ma_df)}')
print('M-ALERT columns:', list(ma_df.columns))

# Sample n parallel prompt rows; for each row, read the prompt in each language column.
ma_sample_rows = ma_df.sample(
    n=min(REPRO['n_per_language'], len(ma_df)),
    random_state=REPRO['seed'],
).reset_index(drop=True)

ma_samples = []
for lang in REPRO['malert_languages']:
    if lang not in ma_sample_rows.columns:
        print(f'WARNING: M-ALERT has no column for lang={lang}')
        continue
    sub = ma_sample_rows[[lang, 'id', 'category']].copy()
    sub.columns = ['prompt', 'malert_id', 'category']
    sub['lang'] = lang
    sub['dataset'] = 'malert'
    ma_samples.append(sub.reset_index(drop=True))
ma_samples = pd.concat(ma_samples, ignore_index=True)
print(f'M-ALERT: sampled {len(ma_samples)} prompts total ({REPRO["n_per_language"]} parallel rows x {len(REPRO["malert_languages"])} languages)')


In [ ]:
queue = pd.concat([
    pg_samples[['prompt', 'lang', 'dataset']],
    ma_samples[['prompt', 'lang', 'dataset']],
], ignore_index=True)
print(f'Total prompts to evaluate: {len(queue)}')
print(queue.groupby(['dataset', 'lang']).size().unstack(fill_value=0))


## Generation + judging loop

For each model: generate responses for every prompt in the queue, then have Llama Guard 3-1B judge each (prompt, response) pair. Save partial results per-model so a Colab disconnect can be resumed without re-running prior models.


In [ ]:
MAX_NEW_TOKENS = REPRO['max_new_tokens']

def load_model(model_id, *, dtype=torch.bfloat16):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype, device_map='auto',
    )
    model.eval()
    return model, tok

def _chat_to_inputs(tok, messages, *, add_generation_prompt=True, device='cuda'):
    """Tokenise a chat-formatted message list, returning a kwargs dict for model.generate().

    Newer Qwen tokenizers return a BatchEncoding (dict-like) from apply_chat_template
    even when return_tensors='pt' is set, which model.generate() can't accept positionally.
    This helper normalises to a {'input_ids': tensor, ...} kwargs dict that always works.
    """
    out = tok.apply_chat_template(
        messages,
        add_generation_prompt=add_generation_prompt,
        return_tensors='pt',
        return_dict=True,
    )
    if isinstance(out, dict) or hasattr(out, 'data'):
        return {k: v.to(device) for k, v in (out.items() if hasattr(out, 'items') else out.data.items())}
    return {'input_ids': out.to(device)}

def generate(model, tok, prompt, *, do_sample=False):
    msgs = [{'role': 'user', 'content': prompt}]
    inputs = _chat_to_inputs(tok, msgs, device=model.device)
    input_len = inputs['input_ids'].shape[1]
    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
    if do_sample:
        gen_kwargs.update(dict(do_sample=True, temperature=0.7))
    else:
        gen_kwargs.update(dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    return tok.decode(out[0, input_len:], skip_special_tokens=True).strip()

def llama_guard_judge(guard_model, guard_tok, prompt, response):
    chat = [
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': response},
    ]
    inputs = _chat_to_inputs(guard_tok, chat, add_generation_prompt=False, device=guard_model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = guard_model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=guard_tok.pad_token_id)
    label = guard_tok.decode(out[0, input_len:], skip_special_tokens=True).strip().lower()
    head = label.split('\n', 1)[0].strip()
    if head.startswith('unsafe'):
        return 'unsafe'
    if head == 'safe':
        return 'safe'
    return 'unknown'

print('Loading judge: Llama Guard 3-1B...')
guard_model, guard_tok = load_model(REPRO['judge'])
print('Judge loaded.')


In [ ]:
all_results = []
for model_id in REPRO['models']:
    short = model_id.split('/')[-1].lower()
    partial_path = RESULTS_DIR / f'multilingual_partial_{short}.json'
    if partial_path.exists():
        print(f'[skip] {model_id}: partial exists at {partial_path.name}')
        all_results.extend(json.loads(partial_path.read_text()))
        continue

    print(f'=== {model_id} ===')
    do_sample = 'qwen3' in short
    model, tok = load_model(model_id)

    rows = []
    for i, item in enumerate(queue.itertuples(index=False)):
        try:
            response = generate(model, tok, item.prompt, do_sample=do_sample)
            label = llama_guard_judge(guard_model, guard_tok, item.prompt, response)
        except Exception as exc:
            response = f'<<ERROR: {exc}>>'
            label = 'unknown'
        rows.append({
            'model': model_id, 'dataset': item.dataset, 'lang': item.lang,
            'prompt': item.prompt, 'response': response, 'label': label,
        })
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(queue)}')

    partial_path.write_text(json.dumps(rows, indent=2))
    print(f'Saved partial -> {partial_path.name}')
    all_results.extend(rows)
    del model; gc.collect(); torch.cuda.empty_cache()

print(f'Total results: {len(all_results)}')


## Aggregate ASR with 95% Wilson CIs

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def wilson(k, n, alpha=0.05):
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    lo, hi = proportion_confint(k, n, alpha=alpha, method='wilson')
    return p, lo, hi

results_df = pd.DataFrame(all_results)
agg_rows = []
for (model, dataset, lang), g in results_df.groupby(['model', 'dataset', 'lang']):
    n = len(g)
    n_unsafe = int((g['label'] == 'unsafe').sum())
    n_unknown = int((g['label'] == 'unknown').sum())
    p, lo, hi = wilson(n_unsafe, n)
    agg_rows.append({
        'model': model, 'dataset': dataset, 'lang': lang,
        'n': int(n), 'unsafe': n_unsafe, 'unknown': n_unknown,
        'asr': float(p), 'ci_lo': float(lo), 'ci_hi': float(hi),
    })
agg_df = pd.DataFrame(agg_rows)
out_csv = RESULTS_DIR / 'multilingual_polyguard_malert.csv'
out_json = RESULTS_DIR / 'multilingual_polyguard_malert.json'
agg_df.to_csv(out_csv, index=False)
out_json.write_text(json.dumps({'repro': REPRO, 'aggregate': agg_rows, 'raw': all_results}, indent=2))
print(f'Saved {out_csv}')
print(f'Saved {out_json}')
agg_df.sort_values(['dataset', 'model', 'lang'])
